# Image Classification: ANN vs CNN

This notebook compares a baseline ANN and a CNN on the Intel Image Classification dataset.

## Dataset

Download the Intel Image Classification dataset from Kaggle and set `DATA_DIR` to the extracted folder.

In [ ]:
!pip install -q numpy pandas matplotlib torch torchvision

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
sys.path.append(str(PROJECT_DIR))

from main import (
    set_seed, get_device, create_dataloaders,
    ANNClassifier, CNNClassifier, train_model,
    count_trainable_parameters, plot_training_history,
)

DATA_DIR = 'path/to/intel-image-classification'
OUTPUT_DIR = 'outputs'
EPOCHS = 10
BATCH_SIZE = 64

set_seed()
device = get_device()
print('Device:', device)

In [ ]:
train_loader, test_loader, class_names = create_dataloaders(
    DATA_DIR,
    batch_size=BATCH_SIZE,
)
print('Classes:', class_names)
print('Training images:', len(train_loader.dataset))
print('Validation images:', len(test_loader.dataset))

## ANN Model

The ANN uses only fully connected layers after flattening the image.

In [ ]:
ann = ANNClassifier(len(class_names)).to(device)
ann_history, ann_time = train_model(
    ann, train_loader, test_loader, device, epochs=EPOCHS
)
plot_training_history(ann_history, 'ANN', OUTPUT_DIR)
print('ANN parameters:', count_trainable_parameters(ann))
print('ANN training time:', round(ann_time, 2), 'seconds')

import pandas as pd
## CNN Model

The CNN uses convolution, ReLU, max-pooling, and dropout layers.

In [ ]:
cnn = CNNClassifier(len(class_names)).to(device)
cnn_history, cnn_time = train_model(
    cnn, train_loader, test_loader, device, epochs=EPOCHS
)
plot_training_history(cnn_history, 'CNN', OUTPUT_DIR)
print('CNN parameters:', count_trainable_parameters(cnn))
print('CNN training time:', round(cnn_time, 2), 'seconds')

In [ ]:
comparison = pd.DataFrame([
    {
        'Model': 'ANN',
        'Training Time (seconds)': round(ann_time, 2),
        'Validation Accuracy': round(float(ann_history['validation_accuracy'].iloc[-1]), 4),
        'Trainable Parameters': count_trainable_parameters(ann),
    },
    {
        'Model': 'CNN',
        'Training Time (seconds)': round(cnn_time, 2),
        'Validation Accuracy': round(float(cnn_history['validation_accuracy'].iloc[-1]), 4),
        'Trainable Parameters': count_trainable_parameters(cnn),
    },
])
comparison

## Conclusion

The ANN treats image pixels as independent input features. The CNN preserves local spatial relationships and learns visual patterns through shared convolutional filters. This makes CNNs generally more appropriate for image classification.